In [ ]:
%pip install -q huggingface_hub pyarrow pandas matplotlib requests numpy

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
ASSET       = "BTCUSDT"              # uppercase asset symbol
LOOK_BACK   = 1440                   # number of past candles (includes last_candle)
LOOK_AHEAD  = 240                    # number of future candles
DATETIME    = "2025-12-12 20:00:00" # current_time: moment after last_candle closes (UTC)
WINDOW_MODE = "exclusive"            # "exclusive" | "inclusive"
NORM_MODE   = "clip"                 # "clip" | "tanh"
K           = 100                    # k=100  →  ratio 0.01 maps to 1.0

BINS_COUNT  = 200                    # histogram bins for KDE (price axis, spans [-1, 1])
BANDWIDTH   = 5                      # kernel half-width in bins
KERNEL      = "Triangular"           # "Triangular" | "Epanechnikov" | "Uniform"

In [ ]:
import io, re, requests
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import HfApi
from datetime import datetime, timedelta, timezone

REPO    = "https://huggingface.co/datasets/payamdavaee/candles/resolve/main"
REPO_ID = "payamdavaee/candles"

def load_range(asset, start, end, columns=None):
    api      = HfApi()
    files    = api.list_repo_files(REPO_ID, repo_type="dataset")
    asset_lc = asset.lower()
    pattern  = re.compile(rf"^{asset_lc}/{asset_lc}-1m-(\d{{4}})-(\d{{2}})\.parquet$")
    start_ts = int(datetime.fromisoformat(start).replace(tzinfo=timezone.utc).timestamp() * 1000)
    end_ts   = int(datetime.fromisoformat(end  ).replace(tzinfo=timezone.utc).timestamp() * 1000) + 86_400_000
    frames   = []
    for f in sorted(files):
        m = pattern.match(f)
        if not m:
            continue
        resp = requests.get(f"{REPO}/{f}", timeout=60)
        resp.raise_for_status()
        tbl  = pq.read_table(io.BytesIO(resp.content), columns=columns or None)
        df   = tbl.to_pandas()
        ts   = df["ts"].astype("int64")
        df   = df[(ts >= start_ts) & (ts < end_ts)]
        if not df.empty:
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames).sort_values("ts").reset_index(drop=True)


# ── Derive window boundaries from DATETIME ────────────────────────────────────
current_time     = datetime.fromisoformat(DATETIME).replace(tzinfo=timezone.utc)
last_candle_time = current_time - timedelta(minutes=1)           # open time of last_candle
lb_start_time    = last_candle_time - timedelta(minutes=LOOK_BACK - 1)
lb_end_time      = last_candle_time
la_start_time    = current_time
la_end_time      = current_time + timedelta(minutes=LOOK_AHEAD - 1)

load_start = lb_start_time.strftime("%Y-%m-%d")
load_end   = (la_end_time + timedelta(days=1)).strftime("%Y-%m-%d")

print(f"Loading {ASSET} from {load_start} to {load_end} ...")
df_all = load_range(ASSET, load_start, load_end, columns=["ts", "vwap", "v"])
print(f"Loaded {len(df_all):,} candles")

ts_int      = df_all["ts"].astype("int64")
lb_start_ms = int(lb_start_time.timestamp() * 1000)
lb_end_ms   = int(lb_end_time.timestamp()   * 1000)
la_start_ms = int(la_start_time.timestamp() * 1000)
la_end_ms   = int(la_end_time.timestamp()   * 1000)

df_lb = df_all[(ts_int >= lb_start_ms) & (ts_int <= lb_end_ms)].reset_index(drop=True)
df_la = df_all[(ts_int >= la_start_ms) & (ts_int <= la_end_ms)].reset_index(drop=True)

assert len(df_lb) == LOOK_BACK,  f"Expected {LOOK_BACK} look-back candles, got {len(df_lb)}"
assert len(df_la) == LOOK_AHEAD, f"Expected {LOOK_AHEAD} look-ahead candles, got {len(df_la)}"

# ── Daily volume normalization ────────────────────────────────────────────────
# v_daily_avg = mean of all LOOK_BACK volumes → 1-day average at 1-min resolution
v_daily_avg = df_lb["v"].mean()
v_lb_norm   = df_lb["v"].values / v_daily_avg   # shape (LOOK_BACK,)
v_la_norm   = df_la["v"].values / v_daily_avg   # shape (LOOK_AHEAD,)  same normalizer

print(f"Look-back  : {len(df_lb)} candles  ✓")
print(f"Look-ahead : {len(df_la)} candles  ✓")
print(f"v_daily_avg: {v_daily_avg:.4f}")

In [ ]:
# ── Border datetimes ──────────────────────────────────────────────────────────
def fmt_ms(ts_ms):
    return pd.Timestamp(int(ts_ms), unit="ms", tz="UTC").strftime("%Y-%m-%d %H:%M:%S UTC")

lb_first_ts = df_lb["ts"].astype("int64").iloc[0]
lb_last_ts  = df_lb["ts"].astype("int64").iloc[-1]
la_first_ts = df_la["ts"].astype("int64").iloc[0]
la_last_ts  = df_la["ts"].astype("int64").iloc[-1]

print("─" * 52)
print(f"  current_time       :  {DATETIME} UTC")
print(f"  last_candle (open) :  {fmt_ms(lb_last_ts)}")
print("─" * 52)
print(f"  look-back  start   :  {fmt_ms(lb_first_ts)}")
print(f"  look-back  end     :  {fmt_ms(lb_last_ts)}")
print(f"  look-ahead start   :  {fmt_ms(la_first_ts)}")
print(f"  look-ahead end     :  {fmt_ms(la_last_ts)}")
print("─" * 52)

In [ ]:
# ── VWAP + volume charts (2-row grid, volume shares time axis with VWAP) ──────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

x_lb  = np.arange(LOOK_BACK)
x_la  = np.arange(LOOK_AHEAD)
ts_lb = pd.to_datetime(df_lb["ts"].astype("int64"), unit="ms", utc=True)
ts_la = pd.to_datetime(df_la["ts"].astype("int64"), unit="ms", utc=True)

# Shared price y-range across both windows
vwap_all = np.concatenate([df_lb["vwap"].values, df_la["vwap"].values])
v_margin = (vwap_all.max() - vwap_all.min()) * 0.05
price_ylim = (vwap_all.min() - v_margin, vwap_all.max() + v_margin)

# Shared volume y-range across both windows
vol_ylim = (0, max(v_lb_norm.max(), v_la_norm.max()) * 1.1)

fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(
    2, 2,
    width_ratios=[LOOK_BACK, LOOK_AHEAD],
    height_ratios=[3, 1],
    hspace=0.06,
    wspace=0.04,
)

ax_lb_p   = fig.add_subplot(gs[0, 0])
ax_la_p   = fig.add_subplot(gs[0, 1])
ax_lb_vol = fig.add_subplot(gs[1, 0], sharex=ax_lb_p)
ax_la_vol = fig.add_subplot(gs[1, 1], sharex=ax_la_p)

# ── Look-back VWAP ────────────────────────────────────────────────────────────
ax_lb_p.plot(x_lb, df_lb["vwap"].values, color="#26a69a", linewidth=0.9)
ax_lb_p.set_ylim(*price_ylim)
ax_lb_p.set_xlim(-1, LOOK_BACK)
ax_lb_p.set_title(f"{ASSET} — Look-Back ({LOOK_BACK} candles)  [past]", fontsize=9)
ax_lb_p.tick_params(axis="y", labelsize=7)
plt.setp(ax_lb_p.get_xticklabels(), visible=False)

# ── Look-ahead VWAP ───────────────────────────────────────────────────────────
ax_la_p.plot(x_la, df_la["vwap"].values, color="#ef5350", linewidth=0.9)
ax_la_p.set_ylim(*price_ylim)
ax_la_p.set_xlim(-1, LOOK_AHEAD)
ax_la_p.set_title(f"{ASSET} — Look-Ahead ({LOOK_AHEAD} candles)  [future]", fontsize=9)
ax_la_p.yaxis.tick_right()
ax_la_p.yaxis.set_label_position("right")
ax_la_p.tick_params(axis="y", labelsize=7)
plt.setp(ax_la_p.get_xticklabels(), visible=False)

# ── Look-back volume ──────────────────────────────────────────────────────────
ax_lb_vol.bar(x_lb, v_lb_norm, color="#26a69a", width=1.0, linewidth=0, alpha=0.8)
ax_lb_vol.set_ylim(*vol_ylim)
ax_lb_vol.set_xlim(-1, LOOK_BACK)
ax_lb_vol.set_ylabel("norm vol", fontsize=7)
ax_lb_vol.tick_params(axis="y", labelsize=6)
step_lb = max(1, LOOK_BACK // 6)
ax_lb_vol.set_xticks(x_lb[::step_lb])
ax_lb_vol.set_xticklabels(
    [ts_lb.iloc[i].strftime("%H:%M") for i in x_lb[::step_lb]], fontsize=6
)

# ── Look-ahead volume ─────────────────────────────────────────────────────────
ax_la_vol.bar(x_la, v_la_norm, color="#ef5350", width=1.0, linewidth=0, alpha=0.8)
ax_la_vol.set_ylim(*vol_ylim)
ax_la_vol.set_xlim(-1, LOOK_AHEAD)
ax_la_vol.yaxis.tick_right()
ax_la_vol.yaxis.set_label_position("right")
ax_la_vol.set_ylabel("norm vol", fontsize=7)
ax_la_vol.tick_params(axis="y", labelsize=6)
step_la = max(1, LOOK_AHEAD // 6)
ax_la_vol.set_xticks(x_la[::step_la])
ax_la_vol.set_xticklabels(
    [ts_la.iloc[i].strftime("%H:%M") for i in x_la[::step_la]], fontsize=6
)

plt.suptitle(f"VWAP + Normalized Volume  |  current_time: {DATETIME} UTC", fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Price normalization + time normalization + KDE ────────────────────────────

# ── Price normalization  (idea_normalize_based_on_last_price_clip) ────────────
price_l = df_lb["vwap"].values[-1]   # last candle of look-back window — anchor

raw_lb = (df_lb["vwap"].values - price_l) / price_l
raw_la = (df_la["vwap"].values - price_l) / price_l

if NORM_MODE == "clip":
    scaled_lb = np.clip(K * raw_lb, -1.0, 1.0)
    scaled_la = np.clip(K * raw_la, -1.0, 1.0)
elif NORM_MODE == "tanh":
    scaled_lb = np.tanh(K * raw_lb)
    scaled_la = np.tanh(K * raw_la)
else:
    raise ValueError(f"Unknown NORM_MODE: {NORM_MODE!r}")

# ── Time normalization  (idea_normalize_based_on_last_price_clip) ─────────────
# t_la starts at 1.0: first look-ahead candle opens exactly at current_time
t    = np.arange(LOOK_BACK) / LOOK_BACK                          # [0, ..., 1439/1440]
t_la = (LOOK_BACK + np.arange(LOOK_AHEAD)) / LOOK_BACK           # [1.0, ..., 1679/1440]

# ── Volume-weighted KDE over normalized look-back prices ──────────────────────
# Histogram: bins span [-1, 1], weights = normalized volume (keeps volume units)
counts, bin_edges = np.histogram(
    scaled_lb,
    bins=BINS_COUNT,
    range=(-1.0, 1.0),
    weights=v_lb_norm,
)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width   = bin_edges[1] - bin_edges[0]

def make_kernel(kernel_type, bandwidth):
    """Symmetric kernel of half-width `bandwidth` bins, normalized to sum=1."""
    x = np.arange(-bandwidth, bandwidth + 1, dtype=float)
    if kernel_type == "Triangular":
        k = np.maximum(1.0 - np.abs(x) / bandwidth, 0.0)
    elif kernel_type == "Epanechnikov":
        u = x / bandwidth
        k = np.maximum(1.0 - u ** 2, 0.0)
    elif kernel_type == "Uniform":
        k = np.ones(len(x))
    else:
        raise ValueError(f"Unknown kernel: {kernel_type!r}")
    k /= k.sum()   # conservation of mass
    return k

kernel_arr = make_kernel(KERNEL, BANDWIDTH)
kde        = np.convolve(counts, kernel_arr, mode="same")

print(f"price_l        = {price_l:.6f}")
print(f"k              = {K}   (ratio {1/K:.4f} → 1.0)")
print(f"t[0]           = {t[0]:.6f}   first look-back candle")
print(f"t[-1]          = {t[-1]:.6f}   last look-back candle (last_candle)")
print(f"current_time   = 1.000000   (t=1.0, not in array)")
print(f"t_la[0]        = {t_la[0]:.6f}   first look-ahead candle (opens at current_time)")
print(f"t_la[-1]       = {t_la[-1]:.6f}   last look-ahead candle")
print(f"scaled_lb      [{scaled_lb.min():.4f}, {scaled_lb.max():.4f}]")
print(f"scaled_la      [{scaled_la.min():.4f}, {scaled_la.max():.4f}]")
print(f"KDE total mass = {kde.sum():.4f}  (raw histogram mass = {counts.sum():.4f})")

In [ ]:
# ── KDE + normalized VWAP charts (3-column, shared price y-axis) ──────────────
# Layout: [KDE (leftmost)] | [look-back norm] | [look-ahead norm]
# KDE y-axis (price) is shared with both time-series charts.

fig = plt.figure(figsize=(14, 3))
gs  = gridspec.GridSpec(
    1, 3,
    width_ratios=[LOOK_AHEAD, LOOK_BACK, LOOK_AHEAD],
    wspace=0.04,
)

ax_kde = fig.add_subplot(gs[0])
ax_lb  = fig.add_subplot(gs[1], sharey=ax_kde)
ax_la  = fig.add_subplot(gs[2], sharey=ax_kde)

# ── KDE (horizontal bars: x = volume, y = normalized price) ──────────────────
ax_kde.barh(
    bin_centers, kde,
    height=bin_width * 0.95,
    color="#26a69a", alpha=0.85, linewidth=0,
)
ax_kde.axhline(0,  color="#aaaaaa", linewidth=0.5, linestyle="--")
ax_kde.axhline( 1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_kde.axhline(-1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_kde.set_ylim(-1.08, 1.08)
ax_kde.set_ylabel("scaled vwap", fontsize=8)
ax_kde.set_xlabel("vol (daily avg units)", fontsize=7)
ax_kde.set_title(
    f"Vol-Weighted KDE\n(kernel={KERNEL}, bw={BANDWIDTH})",
    fontsize=9,
)
ax_kde.tick_params(labelsize=7)

# ── Look-back normalized VWAP ─────────────────────────────────────────────────
ax_lb.plot(t, scaled_lb, color="#26a69a", linewidth=0.8)
ax_lb.axhline(0,  color="#aaaaaa", linewidth=0.5, linestyle="--")
ax_lb.axhline( 1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_lb.axhline(-1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_lb.set_xlabel("normalized time  t = i / look_back", fontsize=8)
ax_lb.set_title(f"Look-Back — normalized VWAP  (k={K}, mode={NORM_MODE})", fontsize=9)
ax_lb.tick_params(axis="x", labelsize=7)
plt.setp(ax_lb.get_yticklabels(), visible=False)  # price axis already on KDE

# ── Look-ahead normalized VWAP ────────────────────────────────────────────────
ax_la.plot(t_la, scaled_la, color="#ef5350", linewidth=0.8)
ax_la.axhline(0,  color="#aaaaaa", linewidth=0.5, linestyle="--")
ax_la.axhline( 1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_la.axhline(-1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_la.set_xlabel("normalized time  t = (look_back + j) / look_back", fontsize=8)
ax_la.yaxis.tick_right()
ax_la.yaxis.set_label_position("right")
ax_la.set_ylabel("scaled vwap", fontsize=8)
ax_la.set_title("Look-Ahead — normalized VWAP  (same price_l anchor)", fontsize=9)
ax_la.tick_params(axis="x", labelsize=7)
ax_la.tick_params(axis="y", labelsize=7)

plt.suptitle(
    f"KDE + Normalized VWAP  |  price_l={price_l:.4f}   k={K}   current_time={DATETIME} UTC",
    fontsize=10,
    y=1.01,
)
plt.tight_layout()
plt.show()